# Clase 184 — BCa bootstrap + permutation test modernos

BCa (bias-corrected accelerated) corrige sesgo y asimetría del bootstrap percentil. Permutation test = inferencia exacta sin asumir distribución.
Requiere: `pip install numpy scipy`.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
# Distribución sesgada (lognormal) — donde percentil bootstrap subperforma
data = rng.lognormal(mean=1.0, sigma=0.6, size=80)
true_mean = np.exp(1.0 + 0.6**2/2)  # E[lognormal]
print(f'sample mean = {data.mean():.3f}  |  true mean = {true_mean:.3f}')

## Bootstrap percentil (baseline)

In [ ]:
B = 5000
boot_means = np.array([rng.choice(data, len(data), replace=True).mean() for _ in range(B)])
lo_pct, hi_pct = np.percentile(boot_means, [2.5, 97.5])
print(f'Percentil 95% CI: [{lo_pct:.3f}, {hi_pct:.3f}]')

## BCa desde scratch
1. **z0** = bias-correction = $\Phi^{-1}(\#\{\hat\theta^*_b < \hat\theta\}/B)$.
2. **a** = acceleration via jackknife: $a = \dfrac{\sum (\bar\theta_{(\cdot)} - \theta_{(i)})^3}{6 [\sum (\bar\theta_{(\cdot)} - \theta_{(i)})^2]^{3/2}}$.
3. Ajuste de los percentiles.

In [ ]:
from scipy.stats import norm

def bca_ci(data, stat_fn, boot_stats, alpha=0.05):
    theta_hat = stat_fn(data)
    z0 = norm.ppf(np.mean(boot_stats < theta_hat))
    # jackknife
    n = len(data)
    jk = np.array([stat_fn(np.delete(data, i)) for i in range(n)])
    jk_mean = jk.mean()
    num = np.sum((jk_mean - jk)**3)
    den = 6 * (np.sum((jk_mean - jk)**2))**1.5
    a = num / den if den > 0 else 0.0
    z_lo, z_hi = norm.ppf(alpha/2), norm.ppf(1 - alpha/2)
    alpha1 = norm.cdf(z0 + (z0 + z_lo) / (1 - a*(z0 + z_lo)))
    alpha2 = norm.cdf(z0 + (z0 + z_hi) / (1 - a*(z0 + z_hi)))
    return np.percentile(boot_stats, [100*alpha1, 100*alpha2])

lo_bca, hi_bca = bca_ci(data, np.mean, boot_means)
print(f'BCa  95% CI: [{lo_bca:.3f}, {hi_bca:.3f}]')
print(f'Pct  95% CI: [{lo_pct:.3f}, {hi_pct:.3f}]')
print(f'true mean  : {true_mean:.3f}  -> BCa suele cubrir mejor con datos sesgados')

## scipy.stats.bootstrap(method='BCa')

In [ ]:
res = stats.bootstrap((data,), np.mean, method='BCa', n_resamples=5000, random_state=42)
print(f'scipy BCa CI: [{res.confidence_interval.low:.3f}, {res.confidence_interval.high:.3f}]')
print(f'manual BCa  : [{lo_bca:.3f}, {hi_bca:.3f}]')

## Permutation test — diferencia de medias
Bajo H0 (mismas distribuciones), los labels son intercambiables. Distribuimos diff_means bajo H0 mezclando labels.

In [ ]:
rng2 = np.random.default_rng(42)
g1 = rng2.normal(0, 1, 40)
g2 = rng2.normal(0.5, 1, 40)
obs = g2.mean() - g1.mean()

pool = np.concatenate([g1, g2])
n1 = len(g1)
B = 10_000
perm_diffs = np.empty(B)
for i in range(B):
    p = rng2.permutation(pool)
    perm_diffs[i] = p[n1:].mean() - p[:n1].mean()
p_manual = (np.sum(np.abs(perm_diffs) >= abs(obs)) + 1) / (B + 1)
print(f'obs diff = {obs:.3f}')
print(f'p (manual)  = {p_manual:.4f}')

In [ ]:
def stat(a, b, axis):
    return a.mean(axis=axis) - b.mean(axis=axis)
res = stats.permutation_test((g2, g1), stat, n_resamples=10_000, alternative='two-sided', random_state=42)
print(f'scipy p     = {res.pvalue:.4f}')
print(f'manual p    = {p_manual:.4f}')

## Cobertura empírica: ¿el CI 95% realmente cubre 95%?
Simulamos 500 datasets lognormal y vemos qué fracción del CI BCa contiene la media true.

In [ ]:
rng3 = np.random.default_rng(42)
n_sims = 500
cov_bca, cov_pct = 0, 0
for _ in range(n_sims):
    d = rng3.lognormal(1.0, 0.6, 50)
    bm = np.array([rng3.choice(d, 50, replace=True).mean() for _ in range(1000)])
    lo_p, hi_p = np.percentile(bm, [2.5, 97.5])
    lo_b, hi_b = bca_ci(d, np.mean, bm)
    if lo_p <= true_mean <= hi_p: cov_pct += 1
    if lo_b <= true_mean <= hi_b: cov_bca += 1
print(f'Cobertura percentil: {cov_pct/n_sims:.3f}  (target 0.95)')
print(f'Cobertura BCa      : {cov_bca/n_sims:.3f}  (target 0.95)')
print('BCa suele estar mas cerca de 0.95 con datos sesgados.')

## Takeaways
1. **Percentil** subperforma con datos sesgados — CI corrido del valor verdadero.
2. **BCa** corrige sesgo (z0) y asimetría (a, via jackknife) → mejor cobertura empírica.
3. **Permutation test** = p-value exacto bajo H0 de intercambiabilidad — sin asumir normal.
4. Default moderno: `scipy.stats.bootstrap(method='BCa')` y `scipy.stats.permutation_test`.

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Tres ICs** (percentile, basic, BCa) para la mediana de una lognormal.

In [ ]:
# <!--SOL175184-->
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
x = rng.lognormal(0, 1, 100)
cis = {}
for met in ("percentile", "basic", "BCa"):
    ci = stats.bootstrap((x,), np.median, n_resamples=5_000, method=met, random_state=rng).confidence_interval
    cis[met] = (ci.low, ci.high)
    print(f"{met:10s}: [{ci.low:.3f}, {ci.high:.3f}]")
assert all(lo < hi for lo, hi in cis.values())
print("Tres metodos de IC: OK")

<!--SOL175184-->

**Ej. 2 — `z0` a mano.** Sesgo `z0 = Φ⁻¹(#θ*<θ̂ / B)`.

In [ ]:
# <!--SOL175184-->
theta_hat = np.median(x); B = 5_000
theta_star = np.array([np.median(rng.choice(x, x.size, replace=True)) for _ in range(B)])
prop = np.mean(theta_star < theta_hat); z0 = stats.norm.ppf(prop)
print(f"prop(theta*<theta_hat)={prop:.3f}  z0={z0:.4f}")
assert np.isfinite(z0)
print("z0 a mano: OK")

<!--SOL175184-->

**Ej. 3 — Aceleración `a` con jackknife** (leave-one-out).

In [ ]:
# <!--SOL175184-->
n = x.size
jack = np.array([np.median(np.delete(x, i)) for i in range(n)])
jbar = jack.mean()
a_hat = ((jbar - jack)**3).sum() / (6.0 * (((jbar - jack)**2).sum()**1.5))
print(f"aceleracion a = {a_hat:.5f}")
assert np.isfinite(a_hat)
print("Aceleracion via jackknife: OK")

<!--SOL175184-->

**Ej. 4 — Cobertura empírica** `Exp(1)`, n=25: percentile vs BCa (Monte Carlo reducido).

In [ ]:
# <!--SOL175184-->
true_med = np.log(2); n_sim, nn = 200, 25
cov_p = cov_b = 0
for _ in range(n_sim):
    d = rng.exponential(1.0, nn)
    pci = stats.bootstrap((d,), np.median, n_resamples=499, method="percentile", random_state=rng).confidence_interval
    bci = stats.bootstrap((d,), np.median, n_resamples=499, method="BCa", random_state=rng).confidence_interval
    cov_p += pci.low <= true_med <= pci.high
    cov_b += bci.low <= true_med <= bci.high
print(f"percentile={cov_p/n_sim:.1%}  BCa={cov_b/n_sim:.1%} (nominal 95%)")
assert 0 < cov_b <= n_sim
print("Cobertura percentile vs BCa: OK")

<!--SOL175184-->

**Ej. 5 — `permutation_test`** entre dos lognormales con efecto chico.

In [ ]:
# <!--SOL175184-->
g1 = rng.lognormal(0.0, 1.0, 60); g2 = rng.lognormal(0.15, 1.0, 60)
perm = stats.permutation_test((g1, g2), lambda a, b: np.median(a) - np.median(b),
                              n_resamples=5_000, alternative="two-sided", random_state=rng)
print(f"stat obs={perm.statistic:.4f}  p={perm.pvalue:.4f}")
assert 0 <= perm.pvalue <= 1
print("Permutation test entre lognormales: OK")